# TACO sharpa GPU-sim (mjwarp) rollout benchmark — 2026-07-01

Root-cause audit of slow GPU rollouts in RLinf (`taco_sharpa20hz_tp4_raft_*` experiments).

Findings quantified here:
1. Driver 535 (CUDA 12.2) lacks conditional CUDA graph nodes → mjwarp graph capture fails → fallback host-syncs every solver iteration.
2. `put_data(nconmax=...)` is per-world in mjwarp 3.7 → passing `100*num_envs` allocates `num_envs^2` contacts (OOM at 512 envs).
3. Scene XML has no `<option>` → Newton iterations=100/ls=50 defaults get fully unrolled into the CUDA graph.
4. Newton vs CG comparison (colleague hypothesis): Newton converges in ~2.6 iters vs CG ~6.1 and is faster end-to-end — keep Newton.


In [1]:
import subprocess, sys

PY = '/root/RLinf/.venv/bin/python'
BENCH = '/root/RLinf/notebooks/bench_mjwarp_sharpa.py'

def run(args):
    out = subprocess.run([PY, BENCH] + args, capture_output=True, text=True,
                         cwd='/root/RLinf')
    for line in out.stdout.splitlines() + out.stderr.splitlines():
        if line.startswith('  [') or line.startswith('num_envs') \
           or 'conditional graph' in line or 'Failed to allocate' in line:
            print(line)


## 1. Current fallback path (no CUDA graph) vs fixed graph path
`fallback_no_graph_newton_it100` reproduces what `TacoEnvGPU` does today on this driver; the graph cases require `opt.graph_conditional=False`.

In [2]:
run(['--num-envs', '512', '--control-steps', '3', '--cases', 'fallback'])

conditional graph supported: False
num_envs=512 control_steps=3 substeps=25
  [fallback_no_graph_newton_it100] build 4.7s warm 1.4s capture 0.0s |   1175.7 ms/ctrl-step | est h98 rollout   115.2s | solver_niter mean 2.0 max 2 | ncon/env 50 nefc max 213 | nan 0.000
        - conditional graph nodes are not available for < 12.4


In [3]:
run(['--num-envs', '512', '--control-steps', '10',
     '--cases', 'graph_it100,graph_it10,graph_it5'])

conditional graph supported: False
num_envs=512 control_steps=10 substeps=25
  [graph_newton_it100] build 5.3s warm 1.2s capture 0.3s |    292.6 ms/ctrl-step | est h98 rollout    28.7s | solver_niter mean 2.6 max 7 | ncon/env 50 nefc max 213 | nan 0.000
  [graph_newton_it10_ls20] build 1.6s warm 0.1s capture 0.0s |    158.4 ms/ctrl-step | est h98 rollout    15.5s | solver_niter mean 2.5 max 7 | ncon/env 50 nefc max 213 | nan 0.000
  [graph_newton_it5_ls15] build 1.8s warm 0.1s capture 0.0s |    151.6 ms/ctrl-step | est h98 rollout    14.9s | solver_niter mean 2.7 max 5 | ncon/env 50 nefc max 213 | nan 0.000
        - conditional graph nodes are not available for < 12.4


## 2. Newton vs CG (colleague hypothesis 1)
CG needs ~2.3x more iterations to converge on this scene and is slower end-to-end. The 'CG is faster' folklore comes from MJX/JAX, which cannot run Newton's conditional loop efficiently — it does not apply to mjwarp.

In [4]:
run(['--num-envs', '512', '--control-steps', '10', '--cases', 'cg_it10,cg_it25'])

conditional graph supported: False
num_envs=512 control_steps=10 substeps=25
  [graph_cg_it10] build 5.3s warm 1.0s capture 0.2s |    173.6 ms/ctrl-step | est h98 rollout    17.0s | solver_niter mean 6.0 max 10 | ncon/env 49 nefc max 213 | nan 0.000
  [graph_cg_it25] build 1.8s warm 0.0s capture 0.0s |    204.0 ms/ctrl-step | est h98 rollout    20.0s | solver_niter mean 6.0 max 12 | ncon/env 50 nefc max 213 | nan 0.000
        - conditional graph nodes are not available for < 12.4


## 3. nconmax per-world semantics bug
`broken_nconmax` reproduces the old `nconmax=100*num_envs` call. At 512 envs it OOMs (~57 GB single allocation); at 128 envs it 'only' costs ~1.5x speed.

In [5]:
run(['--num-envs', '128', '--control-steps', '10',
     '--cases', 'broken_nconmax,graph_it10'])

conditional graph supported: False
num_envs=128 control_steps=10 substeps=25
  [broken_nconmax_it10_graph] build 4.9s warm 2.0s capture 0.2s |    112.4 ms/ctrl-step | est h98 rollout    11.0s | solver_niter mean 2.6 max 7 | ncon/env 50 nefc max 213 | nan 0.000
  [graph_newton_it10_ls20] build 1.8s warm 0.0s capture 0.0s |    104.4 ms/ctrl-step | est h98 rollout    10.2s | solver_niter mean 2.6 max 5 | ncon/env 50 nefc max 213 | nan 0.000
        - conditional graph nodes are not available for < 12.4


## 4. Scaling + timestep headroom
dt=0.004 halves substeps (25→13) — physics change, needs A/B validation before adoption.

In [6]:
run(['--num-envs', '1024', '--control-steps', '10', '--cases', 'graph_it10'])

conditional graph supported: False
num_envs=1024 control_steps=10 substeps=25
  [graph_newton_it10_ls20] build 5.7s warm 1.3s capture 0.2s |    191.6 ms/ctrl-step | est h98 rollout    18.8s | solver_niter mean 2.6 max 7 | ncon/env 50 nefc max 213 | nan 0.000
        - conditional graph nodes are not available for < 12.4


In [7]:
run(['--num-envs', '512', '--control-steps', '10', '--cases', 'dt4'])

conditional graph supported: False
num_envs=512 control_steps=10 substeps=25
  [graph_newton_it10_dt0.004] build 4.6s warm 1.2s capture 0.2s |     74.8 ms/ctrl-step | est h98 rollout     7.3s | solver_niter mean 2.6 max 6 | ncon/env 47 nefc max 201 | nan 0.000
        - conditional graph nodes are not available for < 12.4


## 5. Fixed `TacoEnvGPU` end-to-end (obs synthesis + reward included)
Uses the real experiment config with `sim_backend=gpu`. Also reports the NaN-world fraction near episode end (mjwarp has no BADQACC auto-reset; rewards are now nan-guarded and logged as `sim_nan_frac`).

In [8]:
out = subprocess.run([PY, '/root/RLinf/notebooks/smoke_taco_env_gpu.py',
                      '--num-envs', '512', '--chunk-steps', '25'],
                     capture_output=True, text=True, cwd='/root/RLinf')
for line in out.stdout.splitlines():
    if any(k in line for k in ('init:', 'reset:', 'rollout:', 'rewards', 'nan')):
        print(line)

init: 9.5s | substeps=25 graph=OK solver_it=10 ls_it=20
reset: 0.44s | obs keys {'states': (512, 4, 56), 'pointcloud': (512, 4, 512, 3), 'tool_pointcloud': (512, 4, 512, 3)}
rollout: 25 chunk steps (100 ctrl steps) in 18.7s -> 187.5 ms/ctrl-step
rewards last chunk mean 0.3215 | episode return mean 71.266 | steps 98
qpos nan frac 0.0035


## 6. RL loop timer comparison (from live runs)
| setup | env_interact_step | generate_rollouts | global step |
|---|---|---|---|
| CPU sim, 512 envs / 5 workers (live RAFT run) | ~161 s | ~195 s | ~260 s |
| GPU sim fixed, 256 envs / 1 GPU (smoke-gpusim-fixed-20260701) | ~15 s | ~31 s | ~67 s |
